# Deep Dive: Sentiment Signals and Escalation Thresholds

This notebook provides mathematical derivations and analysis that supplement the [Session 4 Lecture](../eCornell-AI-Finance-S4-Lecture-ProductionOps-May-2026.ipynb). We derive the exponentially weighted least squares (EWLS) estimator for the single index model, analyze the sentiment signal design, and examine escalation threshold sensitivity.

## 1. EWLS Derivation: From OLS to Exponentially Weighted Estimation

The single index model (SIM) relates each asset's growth rate to the market:

$$g_{i,t} = \alpha_i + \beta_i g_{m,t} + \epsilon_{i,t}$$

where $g_{i,t}$ is the continuously compounded growth rate (CCGR) of asset $i$ on day $t$, $g_{m,t}$ is the market CCGR, $\alpha_i$ is the asset-specific excess return, $\beta_i$ is the market sensitivity, and $\epsilon_{i,t} \sim \mathcal{N}(0, \sigma_i^2)$ is idiosyncratic noise.

### 1.1 The OLS Baseline

> __Ordinary Least Squares.__ The OLS estimator minimizes the sum of squared residuals over $T$ observations:
>
> $$\min_{\alpha, \beta} \sum_{t=1}^{T} \left(g_{i,t} - \alpha - \beta\, g_{m,t}\right)^2$$
>
> Taking partial derivatives and setting them to zero yields the normal equations:
>
> $$\frac{\partial}{\partial \alpha}: \quad -2 \sum_{t=1}^{T} \left(g_{i,t} - \alpha - \beta\, g_{m,t}\right) = 0$$
>
> $$\frac{\partial}{\partial \beta}: \quad -2 \sum_{t=1}^{T} g_{m,t}\left(g_{i,t} - \alpha - \beta\, g_{m,t}\right) = 0$$
>
> The solution is the standard OLS estimator. Every observation receives equal weight, so the estimate reflects the entire sample history equally.

### 1.2 Introducing Exponential Weights

> __Exponential weighting.__ In a regime-switching market, old observations may reflect a different regime than the current one. We assign each observation a weight that decays exponentially with age:
>
> $$w_t = \eta^{T-t}$$
>
> where $\eta = 2^{-1/h}$ and $h$ is the half-life in days. The most recent observation ($t = T$) has weight $w_T = 1$. An observation $h$ days old has weight $w_{T-h} = \eta^h = 1/2$. An observation $2h$ days old has weight $1/4$, and so on.

The EWLS objective replaces the unweighted sum with a weighted sum:

$$\min_{\alpha, \beta} \sum_{t=1}^{T} w_t \left(g_{i,t} - \alpha - \beta\, g_{m,t}\right)^2$$

### 1.3 The EWLS Normal Equations

> __Weighted normal equations.__ Taking partial derivatives of the EWLS objective:
>
> $$\frac{\partial}{\partial \alpha}: \quad \sum_{t=1}^{T} w_t \left(g_{i,t} - \alpha - \beta\, g_{m,t}\right) = 0$$
>
> $$\frac{\partial}{\partial \beta}: \quad \sum_{t=1}^{T} w_t\, g_{m,t} \left(g_{i,t} - \alpha - \beta\, g_{m,t}\right) = 0$$
>
> Define the weighted sufficient statistics:
>
> $$S_w = \sum_{t=1}^{T} w_t, \quad S_x = \sum_{t=1}^{T} w_t\, g_{m,t}, \quad S_y = \sum_{t=1}^{T} w_t\, g_{i,t}$$
>
> $$S_{xx} = \sum_{t=1}^{T} w_t\, g_{m,t}^2, \quad S_{xy} = \sum_{t=1}^{T} w_t\, g_{i,t}\, g_{m,t}$$

Expanding the normal equations in terms of these statistics:

$$S_y - \alpha\, S_w - \beta\, S_x = 0 \quad \Rightarrow \quad \alpha = \frac{S_y - \beta\, S_x}{S_w}$$

$$S_{xy} - \alpha\, S_x - \beta\, S_{xx} = 0$$

Substituting the first into the second:

$$S_{xy} - \frac{S_y - \beta\, S_x}{S_w} \cdot S_x - \beta\, S_{xx} = 0$$

$$S_{xy} - \frac{S_x S_y}{S_w} + \frac{\beta\, S_x^2}{S_w} - \beta\, S_{xx} = 0$$

$$\beta\left(\frac{S_x^2}{S_w} - S_{xx}\right) = \frac{S_x S_y}{S_w} - S_{xy}$$

> __EWLS solution.__ Solving for $\hat{\beta}$ and $\hat{\alpha}$:
>
> $$\boxed{\hat{\beta} = \frac{S_w\, S_{xy} - S_x\, S_y}{S_w\, S_{xx} - S_x^2}}$$
>
> $$\boxed{\hat{\alpha} = \frac{S_y - \hat{\beta}\, S_x}{S_w}}$$
>
> These are identical in form to the OLS normal equations, but the sums are weighted. The denominator $S_w\, S_{xx} - S_x^2$ is the weighted analog of $T \sum g_{m,t}^2 - (\sum g_{m,t})^2$.

### 1.4 The Online Update Rule

> __O(1) updates.__ When a new observation $(g_{i,T+1}, g_{m,T+1})$ arrives, we do not need to recompute the sums from scratch. The exponential weight structure allows an incremental update. Multiplying all previous weights by $\eta$ shifts the decay by one step, then we add the new observation with weight $1$:
>
> $$S_w \leftarrow \eta\, S_w + 1$$
>
> $$S_x \leftarrow \eta\, S_x + g_{m,T+1}$$
>
> $$S_y \leftarrow \eta\, S_y + g_{i,T+1}$$
>
> $$S_{xx} \leftarrow \eta\, S_{xx} + g_{m,T+1}^2$$
>
> $$S_{xy} \leftarrow \eta\, S_{xy} + g_{i,T+1}\, g_{m,T+1}$$
>
> After updating the five sufficient statistics, recompute $\hat{\alpha}$ and $\hat{\beta}$ using the boxed formulas. This is $O(1)$ per update per asset — no matrix inversion is needed for the 2-parameter SIM.

### 1.5 Special Case: Recovering OLS

> __Infinite half-life.__ When $\eta = 1$ (equivalently, $h \to \infty$), every observation has weight $w_t = 1^{T-t} = 1$. The sufficient statistics reduce to:
>
> $$S_w = T, \quad S_x = \sum_{t=1}^{T} g_{m,t}, \quad S_y = \sum_{t=1}^{T} g_{i,t}$$
>
> $$S_{xx} = \sum_{t=1}^{T} g_{m,t}^2, \quad S_{xy} = \sum_{t=1}^{T} g_{i,t}\, g_{m,t}$$
>
> These are the unweighted sums, so the EWLS estimator reduces exactly to OLS. EWLS is a strict generalization of OLS with $\eta$ controlling the memory horizon.

### 1.6 Bias-Variance Tradeoff

> __Short half-life (small $h$).__ A short half-life makes $\eta$ small, so old observations decay quickly. The estimator reacts fast to regime changes but uses fewer effective observations. Estimation variance is high — the estimates can be noisy day-to-day.

> __Long half-life (large $h$).__ A long half-life makes $\eta$ close to $1$, so old observations retain substantial weight. The estimator is stable but slow to adapt. If a regime change occurs, the estimates may lag the true parameters for many days.

The effective sample size of the EWLS estimator is:

$$n_{\text{eff}} = \frac{S_w^2}{\sum_{t=1}^{T} w_t^2} = \frac{\left(\sum \eta^{T-t}\right)^2}{\sum \eta^{2(T-t)}} \approx \frac{1+\eta}{1-\eta} \quad \text{(for large } T \text{)}$$

For a 21-day half-life ($h = 21$): $\eta = 2^{-1/21} \approx 0.9675$, so $n_{\text{eff}} \approx 60.5$. The estimator behaves as if it uses about 60 equally-weighted observations, even though it has access to the entire history.

---

## 2. Sentiment Signal Design

The sentiment score maps recent market returns into a bounded signal that drives the lambda override in the production loop.

### 2.1 The Sentiment Function

> __Definition.__ The sentiment score at time $t$ is:
>
> $$s_t = \tanh\left(G \cdot r_{5d,t}\right)$$
>
> where $r_{5d,t} = (P_t - P_{t-5})/P_{t-5}$ is the 5-day simple return of the market index (SPY) and $G = 10$ is the gain parameter.

### 2.2 Why Tanh?

> __Bounded output.__ The hyperbolic tangent maps $\mathbb{R} \to (-1, 1)$. No matter how extreme the 5-day return, the sentiment score stays bounded. A linear mapping would let a single extreme day dominate the signal.

> __Antisymmetry.__ $\tanh(-x) = -\tanh(x)$. A $+5\%$ weekly return produces the same magnitude sentiment as a $-5\%$ return, with opposite sign. The signal treats bullish and bearish momentum symmetrically.

> __Saturation.__ For large arguments, $\tanh(x) \to \pm 1$. Once the return is extreme enough, the signal saturates — a $-20\%$ crash and a $-30\%$ crash both produce sentiment near $-1$. This prevents the system from distinguishing between "very bad" and "catastrophically bad" at the signal level; that distinction is handled by the escalation triggers.

### 2.3 The Gain Parameter

> __Sensitivity control.__ The gain $G$ controls how quickly the signal saturates. High $G$ makes the signal nearly binary (close to $\pm 1$ for moderate returns). Low $G$ keeps it proportional across a wider range.

With $G = 10$, the mapping produces:

| 5-day return $r_{5d}$ | $G \cdot r_{5d}$ | $s_t = \tanh(G \cdot r_{5d})$ | Interpretation |
|---|---|---|---|
| $+5\%$ | $0.5$ | $0.46$ | Moderate bullish |
| $+2\%$ | $0.2$ | $0.20$ | Mild bullish |
| $-2\%$ | $-0.2$ | $-0.20$ | Mild bearish |
| $-5\%$ | $-0.5$ | $-0.46$ | Moderate bearish |
| $-10\%$ | $-1.0$ | $-0.76$ | Strong bearish |
| $-20\%$ | $-2.0$ | $-0.96$ | Near-saturated bearish |

A typical sentiment threshold of $s_{\text{threshold}} = -0.5$ fires when the 5-day return is approximately $-5.5\%$ or worse. This is a meaningful weekly decline but not an everyday occurrence.

### 2.4 Connection to Momentum Factors

> __Short-horizon momentum.__ The 5-day return is a short-horizon momentum signal. The academic momentum literature (Jegadeesh and Titman, 1993) documents return persistence at 3--12 month horizons. Our 5-day window captures a different effect: intraweek price dynamics that may reflect mean-reversion at the shortest horizons or continuation of institutional order flow over several days.

The sentiment signal is not used for alpha generation. It is a risk management tool: when short-horizon momentum is strongly negative, increase risk aversion. This is closer in spirit to a trend-following overlay than a momentum factor portfolio.

### 2.5 The Override Mechanism

> __Binary switch.__ The sentiment override operates as a binary switch between two modes:
>
> $$\lambda_{\text{effective},t} = \begin{cases} \lambda_t & \text{if } s_t \geq s_{\text{threshold}} \\ \lambda_{\text{override}} & \text{if } s_t < s_{\text{threshold}} \end{cases}$$
>
> where $\lambda_t$ is the EMA-based risk aversion from the engine and $\lambda_{\text{override}}$ is a fixed defensive value (e.g., 2.0).

The system has two operating modes: normal (sentiment above threshold) and defensive (sentiment below threshold). There is no gradual interpolation. This is a deliberate design choice: the Cobb-Douglas allocator already provides continuous adjustment through $\lambda_t$; the override is a discrete safety layer that activates only when momentum is strongly negative.

---

## 3. Escalation Threshold Analysis

The three escalation triggers protect the production system from large losses, but their thresholds must be calibrated. Too tight and the system halts on normal market noise. Too loose and it fails to protect in a genuine crash.

### 3.1 The Drawdown Trigger

> __Definition.__ The drawdown trigger fires when portfolio equity has declined by more than $d_{\max}$ from its peak:
>
> $$\frac{W_{\text{peak}} - W_t}{W_{\text{peak}}} > d_{\max}$$
>
> When this condition is true, the system de-risks to 100% cash and notifies the investment committee.

### 3.2 Connection to Portfolio Insurance (CPPI)

> __Constant Proportion Portfolio Insurance.__ CPPI (Black and Jones, 1987) maintains a floor value $F$ and invests a multiple $m$ of the cushion $(W_t - F)$ in risky assets:
>
> $$\text{Risky allocation} = m \cdot (W_t - F)$$
>
> As portfolio value $W_t$ approaches the floor, the risky allocation shrinks toward zero. If $W_t$ reaches $F$, the portfolio is entirely in cash.

Our drawdown trigger is a simplified CPPI. The floor is implicitly $F = (1 - d_{\max}) \cdot W_{\text{peak}}$. Instead of continuously adjusting the risky allocation as $W_t$ approaches $F$, we make a single binary decision: if $W_t < F$, move to 100% cash. The sentiment override partially fills the role of CPPI's continuous de-risking by increasing $\lambda$ (and thus shifting toward safer assets) when momentum deteriorates.

### 3.3 Threshold Sensitivity

> __The calibration problem.__ The threshold $d_{\max}$ must balance two errors:
>
> | $d_{\max}$ | Risk | Consequence |
> |---|---|---|
> | Too tight (e.g., 5%) | Triggers on normal volatility | Frequent false alarms, missed upside |
> | Too loose (e.g., 40%) | Fails to trigger before large losses | Inadequate capital protection |

To calibrate, we need to estimate how large drawdowns are under normal conditions.

> __Expected maximum drawdown.__ For a driftless Brownian motion with annualized volatility $\sigma$ observed over $T$ trading days, Magdon-Ismail and Atiya (2004) show the expected maximum drawdown scales as:
>
> $$\mathbb{E}[\text{MDD}] \approx \sigma_{\text{daily}} \cdot \sqrt{\frac{\pi T}{4}}$$
>
> where $\sigma_{\text{daily}} = \sigma / \sqrt{252}$. For a portfolio with $\sigma = 15\%$ annualized and $T = 252$ trading days:
>
> $$\sigma_{\text{daily}} = \frac{0.15}{\sqrt{252}} \approx 0.00945$$
>
> $$\mathbb{E}[\text{MDD}] \approx 0.00945 \times \sqrt{\frac{\pi \cdot 252}{4}} \approx 0.00945 \times 14.1 \approx 13.3\%$$

This means a 15% drawdown threshold will trigger roughly once per year in normal markets. A 25% threshold corresponds to roughly $25/13.3 \approx 1.9$ times the expected annual maximum drawdown under Brownian motion, making it a rare event in normal conditions but still protective during genuine crashes.

### 3.4 Practical Threshold Guidance

> __Rules of thumb.__ For a portfolio with annualized volatility $\sigma$:
>
> | Threshold $d_{\max}$ | Approximate trigger frequency | Use case |
> |---|---|---|
> | $\approx \mathbb{E}[\text{MDD}]$ | $\sim$once per year | Tight protection, frequent halts |
> | $\approx 1.5 \times \mathbb{E}[\text{MDD}]$ | $\sim$once every 2--3 years | Moderate protection |
> | $\approx 2 \times \mathbb{E}[\text{MDD}]$ | Rare (tail event) | Loose protection, crash-only |
>
> The Brownian motion approximation understates tail risk because real returns have fat tails. In practice, set $d_{\max}$ conservatively and let the sentiment override handle moderate declines.

### 3.5 The Bandit Churn Limit

> __Definition.__ The bandit churn limit $C_{\max}$ caps the number of asset inclusion changes per day:
>
> $$\sum_{i=1}^{K} \left|a_t^{(i)} - a_{t-1}^{(i)}\right| > C_{\max}$$
>
> where $\mathbf{a}_t \in \{0,1\}^K$ is the bandit's action vector and $K$ is the total number of assets.

> __Calibration.__ If the bandit changes all $K$ assets in one day, the reward landscape has likely shifted dramatically (e.g., due to a regime change or a data error). The threshold should scale with $K$:
>
> $$C_{\max} = \left\lceil K/3 \right\rceil$$
>
> For $K = 10$ assets, $C_{\max} = 4$: the bandit can change up to 4 assets per day without triggering the churn alarm. If it tries to swap 5 or more, the system freezes the previous allocation and logs the event for review.

The churn limit does not prevent the bandit from eventually reaching any action vector. It restricts the rate of change, forcing the bandit to make incremental adjustments rather than wholesale portfolio restructuring in a single day.

---

## Summary

This deep dive covered three components of the Session 4 production system:

**EWLS estimation** generalizes OLS by assigning exponentially decaying weights to past observations. The decay parameter $\eta = 2^{-1/h}$ controls the half-life: shorter half-lives react faster to regime changes but increase estimation variance. The five sufficient statistics $(S_w, S_x, S_y, S_{xx}, S_{xy})$ update in $O(1)$ per observation, making EWLS practical for daily production use.

**Sentiment signal design** uses $\tanh(G \cdot r_{5d})$ to map 5-day market returns into a bounded $(-1, 1)$ score. The $\tanh$ provides saturation (extreme returns do not dominate), antisymmetry (symmetric treatment of rallies and crashes), and boundedness (the signal stays interpretable). The override mechanism is a binary switch: when sentiment drops below threshold, lambda jumps to a defensive value.

**Escalation thresholds** must balance false alarms against missed crashes. The expected maximum drawdown under Brownian motion provides a baseline for calibrating $d_{\max}$. The bandit churn limit scales as $\lceil K/3 \rceil$ to allow gradual portfolio evolution while flagging wholesale restructuring.

---

Return to [Session 4 Lecture](../eCornell-AI-Finance-S4-Lecture-ProductionOps-May-2026.ipynb) or [Example 1](../eCornell-AI-Finance-S4-Example-ProductionPortfolioEngine-May-2026.ipynb).